# HoopStats on Colab (GPU) — SAM2 tracking + portfolio report

Runs the `hoopstats` pipeline on a GPU so the court map uses **SAM2 mask
propagation** (notebook-quality tracking) instead of CPU ByteTrack, then builds
the self-contained portfolio page.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is fine).
Add your keys under the Colab **Secrets** (🔑) panel: `ROBOFLOW_API_KEY` and
`HF_TOKEN`.


## 1. API keys (from Colab Secrets)


In [ ]:
import os
from google.colab import userdata
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('keys set')


## 2. Get the hoopstats repo

Either clone from GitHub (set `REPO_URL`) or mount Google Drive and point
`PROJECT_DIR` at a copy you uploaded.


In [ ]:
REPO_URL = ''  # TODO: e.g. 'https://github.com/youruser/basketball-box-score-ai.git'
PROJECT_DIR = '/content/basketball-box-score-ai'

if REPO_URL:
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    # Drive alternative: mount and copy/point at your uploaded repo
    from google.colab import drive; drive.mount('/content/drive')
    # TODO: set PROJECT_DIR to your repo path under /content/drive/MyDrive/...
%cd {PROJECT_DIR}


## 3. Install SAM2 real-time + checkpoint

Same fork the reference notebook uses; it provides `build_sam2_camera_predictor`.


In [ ]:
%cd /content
!git clone https://github.com/Gy920/segment-anything-2-real-time.git
%cd /content/segment-anything-2-real-time
%pip install -e . -q
!(cd checkpoints && bash download_ckpts.sh)
SAM2_DIR = '/content/segment-anything-2-real-time'


## 4. Install hoopstats runtime deps (GPU)

We install the package without its CPU deps, then the GPU/runtime deps the
tracking + report paths actually use (skips paddleocr, which isn't needed here).


In [ ]:
%cd {PROJECT_DIR}
%pip install -e . --no-deps -q
%pip install -q inference-gpu 'supervision==0.27.0rc4' \
  'git+https://github.com/roboflow/sports.git@feat/basketball' \
  scipy opencv-python python-dotenv tqdm


## 5. Point at SAM2 checkpoint/config and your clip

Use a **short hero clip** (one possession). SAM2 prompts the players on frame 0
and propagates — substitutions / hard camera cuts break that, so keep it short.


In [ ]:
os.environ['SAM2_CHECKPOINT'] = f'{SAM2_DIR}/checkpoints/sam2.1_hiera_large.pt'
os.environ['SAM2_CONFIG'] = 'configs/sam2.1/sam2.1_hiera_l.yaml'  # Hydra config name

VIDEO = '/content/clip.mp4'  # TODO: upload your clip or point at Drive
FRAME = 0                     # frame for the still images (clean half-court view)


## 6. SAM2 court mapping (the hero clip)

`--tracker sam2` runs SAM2 propagation + `clean_paths`. Drop `--max-frames` to do
the whole clip.


In [ ]:
!hoopstats map-court-video --video {VIDEO} --out data/outputs --debug \
  --tracker sam2 \
  --sam2-checkpoint $SAM2_CHECKPOINT --sam2-config $SAM2_CONFIG


## 7. Stage stills for the report


In [ ]:
!hoopstats detect-frame     --video {VIDEO} --out data/outputs --frame {FRAME}
!hoopstats detect-keypoints --video {VIDEO} --out data/outputs --frame {FRAME}
!hoopstats map-court        --video {VIDEO} --out data/outputs --frame {FRAME}


## 8. Build the portfolio report


In [ ]:
import glob
stem = os.path.splitext(os.path.basename(VIDEO))[0]
court = f'data/outputs/court_map_video/{stem}-court-map.mp4'
debug = f'data/outputs/court_map_video/{stem}-tracking-debug.mp4'
!hoopstats report --frames-dir data/outputs \
  --court-video {court} --debug-video {debug} \
  --out data/portfolio/pipeline.html


## 9. Download the portfolio (HTML + assets)


In [ ]:
import shutil
shutil.make_archive('/content/hoopstats_portfolio', 'zip', 'data/portfolio')
from google.colab import files
files.download('/content/hoopstats_portfolio.zip')


### Notes

- If SAM2 errors on the config path, it's a Hydra resolution issue: run with the
  config given as the package-relative name (as above) rather than an absolute
  path, or `%cd` into the SAM2 repo first.
- For the **whole-game** stats (shots/box score) stay on CPU ByteTrack via
  `process-game`; SAM2 here is for the short tracking showcase.
